# 01 · Data Collection

Fetch live BTC and ETH option chains from Deribit, compute independent implied volatilities, cross-check against Deribit's own mark IV, and persist to disk.

**Prerequisites:** internet connection, no API key required (public endpoints only).

**Outputs:**
- `data/raw/btc_surface_<timestamp>.parquet`
- `data/raw/eth_surface_<timestamp>.parquet`

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from src.deribit_client import DeribitClient
from src.iv_calculator import compute_iv_surface, deribit_iv_crosscheck

Path('../data/raw').mkdir(parents=True, exist_ok=True)
Path('../results').mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('VolSurface loaded. Connecting to Deribit...')

In [ ]:
client = DeribitClient()

btc_price = client.get_index_price('btc_usd')
eth_price = client.get_index_price('eth_usd')

print(f'BTC index price : ${btc_price:>12,.2f}')
print(f'ETH index price : ${eth_price:>12,.2f}')
print(f'BTC/ETH ratio   : {btc_price/eth_price:>12.2f}')

In [ ]:
print('Fetching BTC option chain (this takes ~30-60s)...')

btc_raw = client.get_vol_surface_snapshot(
    currency='BTC',
    min_tte_days=1,
    max_tte_years=1.5,
    max_spread_pct=0.40,
)

print(f'\nBTC snapshot: {len(btc_raw)} options across {btc_raw["expiry"].nunique()} expiries')
print('\nExpiry breakdown:')
print(btc_raw.groupby('expiry').size().rename('n_options').to_string())

In [ ]:
print('Computing independent implied volatilities...')

btc = compute_iv_surface(btc_raw, price_col='mid')

n_total  = len(btc)
n_valid  = btc['calc_iv'].notna().sum()
n_failed = n_total - n_valid

print(f'  Total options : {n_total}')
print(f'  IV solved     : {n_valid} ({n_valid/n_total:.1%})')
print(f'  IV failed     : {n_failed} (deep ITM / extreme strikes)')

diff = (btc['calc_iv'] - btc['mark_iv']).abs().dropna()
print(f'\nIV cross-check vs Deribit mark_iv:')
print(f'  Mean |ours - theirs| : {diff.mean()*100:.3f} vol pts')
print(f'  Max  |ours - theirs| : {diff.max()*100:.3f} vol pts')
print(f'  Pct within 1 vol pt  : {(diff < 0.01).mean():.1%}')

In [ ]:
flagged = deribit_iv_crosscheck(btc.dropna(subset=['calc_iv']), tol_abs=0.03)
if flagged.empty:
    print('IV crosscheck: all options within 3 vol pts of Deribit mark_iv.')
else:
    print(f'IV crosscheck: {len(flagged)} options differ by >3 vol pts:')
    print(flagged[['instrument','expiry','strike','type','calc_iv','mark_iv','iv_diff']].to_string())

In [ ]:
print('Fetching ETH option chain...')

eth_raw = client.get_vol_surface_snapshot(
    currency='ETH',
    min_tte_days=1,
    max_tte_years=1.5,
    max_spread_pct=0.40,
)
eth = compute_iv_surface(eth_raw, price_col='mid')

print(f'ETH snapshot: {len(eth)} options across {eth["expiry"].nunique()} expiries')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# ── BTC smiles by expiry ─────────────────────────────────────────────────
ax = axes[0, 0]
btc_valid = btc.dropna(subset=['calc_iv'])
expiries  = sorted(btc_valid['expiry'].unique())
cmap      = plt.cm.viridis(np.linspace(0, 1, len(expiries)))

for exp, color in zip(expiries, cmap):
    grp = btc_valid[btc_valid['expiry'] == exp]
    ax.scatter(grp['log_moneyness'] * 100, grp['calc_iv'] * 100,
               s=20, alpha=0.75, color=color, label=exp)

ax.set_xlabel('Log-moneyness k = ln(K/F) × 100')
ax.set_ylabel('Implied Vol (%)')
ax.set_title('BTC Option Smiles — Live Market', fontweight='bold')
ax.legend(fontsize=7, ncol=2)

# ── Our IV vs Deribit mark_iv ────────────────────────────────────────────
ax = axes[0, 1]
both = btc.dropna(subset=['calc_iv', 'mark_iv'])
ax.scatter(both['mark_iv'] * 100, both['calc_iv'] * 100,
           s=12, alpha=0.5, color='steelblue')
lo = both[['mark_iv', 'calc_iv']].min().min() * 100
hi = both[['mark_iv', 'calc_iv']].max().max() * 100
ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect agreement')
ax.set_xlabel('Deribit mark_iv (%)')
ax.set_ylabel('Our calc_iv (%)')
ax.set_title('IV Cross-check: Ours vs Deribit', fontweight='bold')
ax.legend(fontsize=8)

# ── BTC ATM vol by expiry ────────────────────────────────────────────────
ax = axes[1, 0]
atm = (
    btc_valid
    .assign(abs_lm=lambda d: d['log_moneyness'].abs())
    .sort_values('abs_lm')
    .groupby('expiry')
    .first()
    .reset_index()
    .sort_values('tte')
)
ax.bar(range(len(atm)), atm['calc_iv'] * 100, color='steelblue', alpha=0.85)
ax.set_xticks(range(len(atm)))
ax.set_xticklabels(atm['expiry'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('ATM Implied Vol (%)')
ax.set_title('BTC ATM Vol Term Structure', fontweight='bold')

# ── Relative spread distribution ─────────────────────────────────────────
ax = axes[1, 1]
spreads = btc['rel_spread'].dropna() * 100
ax.hist(spreads, bins=40, color='darkorange', alpha=0.8, edgecolor='white')
ax.axvline(spreads.median(), color='red', lw=1.5, ls='--',
           label=f'Median: {spreads.median():.1f}%')
ax.set_xlabel('Bid-Ask Spread / Mid (%)')
ax.set_ylabel('Count')
ax.set_title('BTC Option Liquidity (Relative Spread)', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('BTC Options — Live Deribit Market Data', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/01_raw_data_overview.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
ts = pd.Timestamp.now('UTC').strftime('%Y%m%d_%H%M')

btc_path = f'../data/raw/btc_surface_{ts}.parquet'
eth_path = f'../data/raw/eth_surface_{ts}.parquet'

btc.to_parquet(btc_path, index=False)
eth.to_parquet(eth_path, index=False)

print(f'Saved BTC surface → {btc_path}  ({len(btc)} rows)')
print(f'Saved ETH surface → {eth_path}  ({len(eth)} rows)')
print(f'\nBTC columns: {list(btc.columns)}')

In [ ]:
print('Sample rows (BTC):')
display(btc[['expiry', 'strike', 'type', 'tte', 'mark_iv', 'calc_iv',
             'delta', 'bid', 'ask', 'open_interest']]
        .dropna(subset=['calc_iv'])
        .head(12)
        .style.format({
            'tte': '{:.4f}',
            'mark_iv': '{:.3f}',
            'calc_iv': '{:.3f}',
            'delta': '{:.3f}',
            'bid': '{:,.2f}',
            'ask': '{:,.2f}',
        }))